In [1]:
import pandas as pd
from datetime import datetime, timedelta
import numpy as np
import warnings
warnings.filterwarnings('ignore')

In [2]:
"""Load training data from Excel file with multiple sheets"""
file_path = "./Phase 1 Training Dataset.xlsx"
xl = pd.ExcelFile(file_path)
groups = xl.sheet_names
train_df = pd.DataFrame()
test_df = pd.DataFrame()
for sheet in groups:
    df = pd.read_excel(file_path, sheet_name=sheet)
    
    # Skip the first row which contains staff type labels
    df = df.iloc[1:]
    
    # Convert the date column to datetime
    df['Date'] = pd.to_datetime(df.iloc[:, 0])
    
    # Sort by date to ensure temporal order
    df = df.sort_values('Date')
    
    # Calculate split point (80% train, 20% test)
    split_idx = int(len(df) * 0.8)
    
    for i in range(5):  # 5 NHs per group
        start_col = i * 4  # Each NH has 4 columns (NH No., CNA, LPN, RN)
        if start_col + 3 >= len(df.columns):
            break
            
        # Extract data for each staff type
        cna_data = pd.to_numeric(df.iloc[:, start_col + 1], errors='coerce')
        lpn_data = pd.to_numeric(df.iloc[:, start_col + 2], errors='coerce')
        rn_data = pd.to_numeric(df.iloc[:, start_col + 3], errors='coerce')        

        num_rows = len(df['Date'])
        # Add the columns in long format to the train and test dataframes
        aux_df_cna =  pd.DataFrame({
            'unique_id': [f"{sheet} NH No. {i + 1} CNA" for _ in range(num_rows)],
            'ds': df['Date'].dropna().tolist(),
            'y': cna_data.dropna().tolist()
        })
        aux_df_lpn =  pd.DataFrame({
            'unique_id': [f"{sheet} NH No. {i + 1} LPN" for _ in range(num_rows)],
            'ds': df['Date'].dropna().tolist(),
            'y': lpn_data.dropna().tolist()
        })
        aux_df_rn = pd.DataFrame({
            'unique_id': [f"{sheet} NH No. {i + 1} RN" for _ in range(num_rows)],
            'ds': df['Date'].dropna().tolist(),
            'y': rn_data.dropna().tolist()
        })
        train_df = pd.concat([train_df, aux_df_cna[:split_idx], aux_df_lpn[:split_idx], aux_df_rn[:split_idx]], ignore_index=True)
        test_df = pd.concat([test_df, aux_df_cna[split_idx:], aux_df_lpn[split_idx:], aux_df_rn[split_idx:]], ignore_index=True)


In [3]:
"""Training the model"""
import logging

import torch
from neuralforecast.core import NeuralForecast
from neuralforecast.models import TSMixer, NBEATS, NHITS, TFT
from neuralforecast.losses.pytorch import MAE, MAPE

logging.getLogger('pytorch_lightning').setLevel(logging.ERROR)
torch.set_float32_matmul_precision('high')

2025-02-25 10:41:23,065	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.
2025-02-25 10:41:23,716	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


In [4]:
Y_df = pd.concat([train_df, test_df], ignore_index=True)
display(Y_df)
# We make validation and test splits
n_time = len(Y_df.ds.unique())
val_size = int(.2 * n_time)
test_size = int(.2 * n_time)
print(len(Y_df.unique_id.unique()))
print(val_size)

,unique_id,ds,y
0,Group 1 NH No. 1 CNA,2024-04-01,87.72
1,Group 1 NH No. 1 CNA,2024-04-02,82.48
2,Group 1 NH No. 1 CNA,2024-04-03,78.47
3,Group 1 NH No. 1 CNA,2024-04-04,93.75
4,Group 1 NH No. 1 CNA,2024-04-05,103.25
...,...,...,...
27295,Group 20 NH No. 5 RN,2024-06-26,12.14
27296,Group 20 NH No. 5 RN,2024-06-27,0.00
27297,Group 20 NH No. 5 RN,2024-06-28,12.75
27298,Group 20 NH No. 5 RN,2024-06-29,8.00


300
18


In [5]:
horizon = 7
input_size = 28
models = [
          # TSMixer(h=horizon,
          #       input_size=input_size,
          #       n_series=300,
          #       max_steps=1000,
          #       val_check_steps=100,
          #       early_stop_patience_steps=5,
          #       scaler_type='identity',
          #       valid_loss=MAE(),
          #       random_seed=12345678,
          #       ),
           NHITS(h=horizon,
                input_size=horizon,
                max_steps=1000,
                val_check_steps=10,
                early_stop_patience_steps=5,
                scaler_type='robust',
                valid_loss=MAE(),
                random_seed=12345678,
                ), 
           # NBEATS(h=horizon,
           #      input_size=horizon,
           #      max_steps=1000,
           #      val_check_steps=100,
           #      early_stop_patience_steps=5,
           #      scaler_type='robust',
           #      valid_loss=MAE(),
           #      random_seed=12345678,
           #      ), 
          # TFT(h=horizon,
          #       input_size=input_size,
          #       batch_size=5,
          #       max_steps=1000,
          #       val_check_steps=100,
          #       early_stop_patience_steps=5,
          #       scaler_type='identity',
          #       valid_loss=MAE(),
          #       random_seed=12345678,
          #       )   
         ]

Seed set to 12345678


In [6]:
nf = NeuralForecast(
    models=models,
    freq='D',
)

Y_hat_df = nf.cross_validation(
    df=Y_df,
    val_size=val_size,
    test_size=test_size,
    n_windows=None
)

Epoch 0: 100%|███████████████████████████████████████| 10/10 [00:02<00:00,  4.56it/s, v_num=174, train_loss_step=2.010]
Validation: |                                                                                    | 0/? [00:00<?, ?it/s]
Validation DataLoader 0: 100%|█████████████████████████████████████████████████████████| 10/10 [00:00<00:00, 35.33it/s]
Epoch 1: 100%|█| 10/10 [00:02<00:00,  4.62it/s, v_num=174, train_loss_step=2.440, valid_loss=10.30, train_loss_epoch=2.
Validation: |                                                                                    | 0/? [00:00<?, ?it/s]
Validation DataLoader 0: 100%|█████████████████████████████████████████████████████████| 10/10 [00:00<00:00, 32.81it/s]
Epoch 2: 100%|█| 10/10 [00:02<00:00,  4.47it/s, v_num=174, train_loss_step=2.210, valid_loss=10.10, train_loss_epoch=2.
Validation: |                                                                                    | 0/? [00:00<?, ?it/s]
Validation DataLoader 0: 100%|██████████

In [7]:
Y_hat_df

,unique_id,ds,cutoff,NHITS,y
0,Group 1 NH No. 1 CNA,2024-06-13,2024-06-12,106.875443,72.95
1,Group 1 NH No. 1 CNA,2024-06-14,2024-06-12,96.249535,98.54
2,Group 1 NH No. 1 CNA,2024-06-15,2024-06-12,101.849541,102.97
3,Group 1 NH No. 1 CNA,2024-06-16,2024-06-12,98.803345,85.11
4,Group 1 NH No. 1 CNA,2024-06-17,2024-06-12,100.624146,91.60
...,...,...,...,...,...
25195,Group 9 NH No. 5 RN,2024-06-26,2024-06-23,29.567356,55.50
25196,Group 9 NH No. 5 RN,2024-06-27,2024-06-23,32.880497,52.74
25197,Group 9 NH No. 5 RN,2024-06-28,2024-06-23,30.991808,27.37
25198,Group 9 NH No. 5 RN,2024-06-29,2024-06-23,28.199728,63.69


In [8]:
from utilsforecast.evaluation import evaluate
from utilsforecast.losses import mae, mape

evaluate(Y_hat_df.drop(columns='cutoff'), metrics=[mae, mape], agg_fn='mean')

,metric,NHITS
0,mae,10.129242
1,mape,0.377335


In [9]:
Y_hat_insample = nf.predict_insample(step_size=1)
display(Y_hat_insample)

Predicting DataLoader 0: 100%|█████████████████████████████████████████████████████████| 10/10 [00:01<00:00,  7.50it/s]


,unique_id,ds,cutoff,NHITS,y
0,Group 1 NH No. 1 CNA,2024-04-01,2024-03-31,-0.071767,87.720001
1,Group 1 NH No. 1 CNA,2024-04-02,2024-03-31,-0.049857,82.480003
2,Group 1 NH No. 1 CNA,2024-04-03,2024-03-31,-0.067341,78.470001
3,Group 1 NH No. 1 CNA,2024-04-04,2024-03-31,-0.154967,93.750000
4,Group 1 NH No. 1 CNA,2024-04-05,2024-03-31,-0.216715,103.250000
...,...,...,...,...,...
140695,Group 9 NH No. 5 RN,2024-06-08,2024-06-05,31.061743,21.360001
140696,Group 9 NH No. 5 RN,2024-06-09,2024-06-05,30.896162,25.719999
140697,Group 9 NH No. 5 RN,2024-06-10,2024-06-05,27.609764,40.599998
140698,Group 9 NH No. 5 RN,2024-06-11,2024-06-05,23.541420,31.580000


In [10]:
Y_hat_agg = Y_hat_df.groupby(['unique_id', 'ds'], as_index=False).agg({
    'NHITS': 'mean',  # Mean of the NHITS column
    'y': 'first'  # Take the first value of 'y'
})
display(Y_hat_agg)
Y_hat_agg_insample = Y_hat_insample.groupby(['unique_id', 'ds'], as_index=False).agg({
    'NHITS': 'mean',  # Mean of the NHITS column
    'y': 'first'  # Take the first value of 'y'
})
display(Y_hat_agg_insample)

,unique_id,ds,NHITS,y
0,Group 1 NH No. 1 CNA,2024-06-13,106.875443,72.95
1,Group 1 NH No. 1 CNA,2024-06-14,91.022079,98.54
2,Group 1 NH No. 1 CNA,2024-06-15,99.185768,102.97
3,Group 1 NH No. 1 CNA,2024-06-16,95.605179,85.11
4,Group 1 NH No. 1 CNA,2024-06-17,94.425377,91.60
...,...,...,...,...
5395,Group 9 NH No. 5 RN,2024-06-26,29.739557,55.50
5396,Group 9 NH No. 5 RN,2024-06-27,32.687092,52.74
5397,Group 9 NH No. 5 RN,2024-06-28,30.707041,27.37
5398,Group 9 NH No. 5 RN,2024-06-29,28.194546,63.69


,unique_id,ds,NHITS,y
0,Group 1 NH No. 1 CNA,2024-04-01,-0.071767,87.720001
1,Group 1 NH No. 1 CNA,2024-04-02,23.126263,82.480003
2,Group 1 NH No. 1 CNA,2024-04-03,34.602047,78.470001
3,Group 1 NH No. 1 CNA,2024-04-04,45.010357,93.750000
4,Group 1 NH No. 1 CNA,2024-04-05,54.114708,103.250000
...,...,...,...,...
21895,Group 9 NH No. 5 RN,2024-06-08,30.133905,21.360001
21896,Group 9 NH No. 5 RN,2024-06-09,29.483833,25.719999
21897,Group 9 NH No. 5 RN,2024-06-10,27.016554,40.599998
21898,Group 9 NH No. 5 RN,2024-06-11,23.671944,31.580000


In [11]:
Y_hat_final = pd.concat([Y_hat_agg_insample, Y_hat_agg], ignore_index=True)
Y_hat_final = Y_hat_final.groupby(['unique_id', 'ds'], as_index=False).mean() 

# Group by 'unique_id'
grouped = Y_hat_final.groupby('unique_id')

# Create a dictionary to store the separate DataFrames
df_dict = {unique_id: group for unique_id, group in grouped}

# Now, df_dict will contain each DataFrame for every unique 'unique_id'

# Iterate through each DataFrame in the dictionary
for unique_id, group in df_dict.items():
    # Replace the first 12 rows of the 'NHITS' column with the 'y' column
    group.iloc[:12, group.columns.get_loc('NHITS')] = group.iloc[:12]['y'].values
    group = group.reset_index(drop=True)
    # After modification, update the DataFrame in the dictionary
    df_dict[unique_id] = group
print(df_dict)

{'Group 1 NH No. 1 CNA':                unique_id         ds       NHITS           y
0   Group 1 NH No. 1 CNA 2024-04-01   87.720001   87.720001
1   Group 1 NH No. 1 CNA 2024-04-02   82.480003   82.480003
2   Group 1 NH No. 1 CNA 2024-04-03   78.470001   78.470001
3   Group 1 NH No. 1 CNA 2024-04-04   93.750000   93.750000
4   Group 1 NH No. 1 CNA 2024-04-05  103.250000  103.250000
..                   ...        ...         ...         ...
86  Group 1 NH No. 1 CNA 2024-06-26   94.061272  101.130000
87  Group 1 NH No. 1 CNA 2024-06-27   96.946037  107.000000
88  Group 1 NH No. 1 CNA 2024-06-28   97.129761  120.750000
89  Group 1 NH No. 1 CNA 2024-06-29   95.895157   95.250000
90  Group 1 NH No. 1 CNA 2024-06-30   92.750755   75.250000

[91 rows x 4 columns], 'Group 1 NH No. 1 LPN':                unique_id         ds      NHITS          y
0   Group 1 NH No. 1 LPN 2024-04-01  57.750000  57.750000
1   Group 1 NH No. 1 LPN 2024-04-02  66.750000  66.750000
2   Group 1 NH No. 1 LPN 2024-04-

In [12]:
# Create a new dictionary to store DataFrames with keys containing both the group number and type of staff eg. "CNA"
joined_dfs_by_group = {} 
for i in range(20): # total groups
    for j in ["CNA", "LPN", "RN"]:
        joined_dfs_by_group[f"Group {i+1} " + j] = {key: df for key, df in df_dict.items() if f"Group {i+1} " in key and j in key}
# iterate through each dictionary containing the results for the same group and staff type, to join the 5 NH values in one dataframe for point estimate and interval calculation
wide_dfs_by_group = {} 
for name, grpstf in joined_dfs_by_group.items():
    # initialise empty DF
    wide_dfs_by_group[name] = pd.DataFrame()
    for k, v in grpstf.items():
        wide_dfs_by_group[name]["ds"] = v["ds"]
        # populate the dataframe with extra columns
        wide_dfs_by_group[name]["NHITS" + k] = v["NHITS"]
        wide_dfs_by_group[name]["y" + k] = v["y"]
    # Calculate the mean across the 5 columns for each row
    wide_dfs_by_group[name]['Mean'] = wide_dfs_by_group[name].filter(like='NHITS').mean(axis=1)
    # Calculate the standard deviation across the 'NHITS' columns
    wide_dfs_by_group[name]['Std'] = wide_dfs_by_group[name].filter(like='NHITS').std(axis=1)
    
    # Calculate the 95% confidence interval using Mean_NHITS as the point estimate
    # wide_dfs_by_group[name]['Lower_95%'] = wide_dfs_by_group[name]['Mean'] - 1.96 * wide_dfs_by_group[name]['Std']
    # wide_dfs_by_group[name]['Upper_95%'] = wide_dfs_by_group[name]['Mean'] + 1.96 * wide_dfs_by_group[name]['Std']
    # Calculate the 95% confidence interval (upper and lower bounds) across the 'NHITS' columns
    wide_dfs_by_group[name]['Lower_95%'] = wide_dfs_by_group[name].filter(like='NHITS').quantile(0.05, axis=1)
    wide_dfs_by_group[name]['Upper_95%'] = wide_dfs_by_group[name].filter(like='NHITS').quantile(0.95, axis=1)

print(wide_dfs_by_group['Group 1 CNA'])

           ds  NHITSGroup 1 NH No. 1 CNA  yGroup 1 NH No. 1 CNA  \
0  2024-04-01                  87.720001              87.720001   
1  2024-04-02                  82.480003              82.480003   
2  2024-04-03                  78.470001              78.470001   
3  2024-04-04                  93.750000              93.750000   
4  2024-04-05                 103.250000             103.250000   
..        ...                        ...                    ...   
86 2024-06-26                  94.061272             101.130000   
87 2024-06-27                  96.946037             107.000000   
88 2024-06-28                  97.129761             120.750000   
89 2024-06-29                  95.895157              95.250000   
90 2024-06-30                  92.750755              75.250000   

    NHITSGroup 1 NH No. 2 CNA  yGroup 1 NH No. 2 CNA  \
0                  166.899994             166.899994   
1                  164.710007             164.710007   
2                  175.2799

In [13]:
def create_prediction_template(output_file, groups):
    """Create predictions for all groups and save to template"""
    # Create Excel writer
    writer = pd.ExcelWriter(output_file, engine='openpyxl')
    df = pd.DataFrame()
    df["Date"] = wide_dfs_by_group['Group 1 CNA']['ds']
    # Create predictions for each group
    for group in groups:
        for staff_type in ['CNA', 'LPN', 'RN']:
            # Select the values corresponding to keys containing both "Group 1" and "CNA"
            for key, value in wide_dfs_by_group.items():
                if f"{group} " in key and staff_type in key:
                    df[f'{staff_type} Point Prediction'] = value["Mean"]
                    df[f'{staff_type} Lower Bound'] = value["Lower_95%"]
                    df[f'{staff_type} Upper Bound'] = value["Upper_95%"]
        
        # Create DataFrame and save to sheet
        df.to_excel(writer, sheet_name=group, index=False)
        
    writer.close()

# create_prediction_template("./Phase 1 Prediction Output.xlsx", groups)

SyntaxError: invalid syntax (2481093358.py, line 13)

In [26]:
def calculate_mae(y_true, y_pred):
    """Calculate Mean Absolute Error"""
    return np.mean(np.abs(y_true - y_pred))

def calculate_mape(y_true, y_pred):
    """Calculate Mean Absolute Percentage Error with handling for small values"""
    # Use a threshold to avoid division by very small numbers
    threshold = 1.0
    mask = y_true > threshold
    if not np.any(mask):
        return np.nan
    return 100 * np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask]))

def calculate_smape(y_true, y_pred):
    """Calculate Symmetric Mean Absolute Percentage Error"""
    return 100 * np.mean(2.0 * np.abs(y_pred - y_true) / (np.abs(y_true) + np.abs(y_pred)))

def calculate_mis(y_true, y_pred, lower_bound, upper_bound):
    """Calculate Mean Interval Score with robust handling of outliers"""
    alpha = 0.05
    n = len(y_true)
    
    # Initialize components of MIS
    coverage_penalty = np.zeros(n)
    width_penalty = np.zeros(n)
    
    # Calculate penalties with outlier-robust handling
    for i in range(n):
        interval_width = upper_bound[i] - lower_bound[i]
        
        if y_true[i] < lower_bound[i]:
            coverage_penalty[i] = 2/alpha * (lower_bound[i] - y_true[i])
        elif y_true[i] > upper_bound[i]:
            coverage_penalty[i] = 2/alpha * (y_true[i] - upper_bound[i])
            
        width_penalty[i] = interval_width
        
    # Use median instead of mean for more robustness
    mis = np.median(width_penalty + coverage_penalty)
    return mis

def evaluate_predictions(y_true, y_pred, lower_bound, upper_bound):
    """Evaluate predictions using all metrics"""
    metrics = {
        'MAE': calculate_mae(y_true, y_pred),
        'MAPE': calculate_mape(y_true, y_pred),
        'SMAPE': calculate_smape(y_true, y_pred),
        'MIS': calculate_mis(y_true, y_pred, lower_bound, upper_bound)
    }
    return metrics

def evaluate_all_data(groups):
    train_metrics = {
        'CNA': {'MAE': [], 'MAPE': [], 'SMAPE': [], 'MIS': []},
        'LPN': {'MAE': [], 'MAPE': [], 'SMAPE': [], 'MIS': []},
        'RN': {'MAE': [], 'MAPE': [], 'SMAPE': [], 'MIS': []}, 
    }
    test_metrics = {
        'CNA': {'MAE': [], 'MAPE': [], 'SMAPE': [], 'MIS': []},
        'LPN': {'MAE': [], 'MAPE': [], 'SMAPE': [], 'MIS': []},
        'RN': {'MAE': [], 'MAPE': [], 'SMAPE': [], 'MIS': []}
    }
    train_metrics_calcs = {
        'y_true_CNA': np.array([]),
        'y_pred_CNA': np.array([]),
        'lower_bound_CNA': np.array([]),
        'upper_bound_CNA': np.array([]),
        'y_true_LPN': np.array([]),
        'y_pred_LPN': np.array([]),
        'lower_bound_LPN': np.array([]),
        'upper_bound_LPN': np.array([]),
        'y_true_RN': np.array([]),
        'y_pred_RN': np.array([]),
        'lower_bound_RN': np.array([]),
        'upper_bound_RN': np.array([])
    }
    test_metrics_calcs = {
        'y_true_CNA': np.array([]),
        'y_pred_CNA': np.array([]),
        'lower_bound_CNA': np.array([]),
        'upper_bound_CNA': np.array([]),
        'y_true_LPN': np.array([]),
        'y_pred_LPN': np.array([]),
        'lower_bound_LPN': np.array([]),
        'upper_bound_LPN': np.array([]),
        'y_true_RN': np.array([]),
        'y_pred_RN': np.array([]),
        'lower_bound_RN': np.array([]),
        'upper_bound_RN': np.array([])
    }
    # Create the arrays for evaulation of each staff group using the metrics
    for group in groups:
        for staff_type in ['CNA', 'LPN', 'RN']:
            # Select the values corresponding to keys containing both "Group 1" and "CNA"
            for key, value in wide_dfs_by_group.items():
                if f"{group} " in key and staff_type in key:
                    # for training set (72 per group staff type, exclude the last 19 rows)
                    train_metrics_calcs[f'y_true_{staff_type}'] = np.concatenate((train_metrics_calcs[f'y_true_{staff_type}'], value[f'y{group} NH No. 1 {staff_type}'][:-19].to_numpy()))
                    train_metrics_calcs[f'y_pred_{staff_type}'] = np.concatenate((train_metrics_calcs[f'y_pred_{staff_type}'], value['Mean'][:-19].to_numpy()))
                    train_metrics_calcs[f'lower_bound_{staff_type}'] = np.concatenate((train_metrics_calcs[f'lower_bound_{staff_type}'], value['Lower_95%'][:-19].to_numpy()))
                    train_metrics_calcs[f'upper_bound_{staff_type}'] = np.concatenate((train_metrics_calcs[f'upper_bound_{staff_type}'], value['Upper_95%'][:-19].to_numpy()))
                    train_metrics_calcs[f'y_true_{staff_type}'] = np.concatenate((train_metrics_calcs[f'y_true_{staff_type}'], value[f'y{group} NH No. 2 {staff_type}'][:-19].to_numpy()))
                    train_metrics_calcs[f'y_pred_{staff_type}'] = np.concatenate((train_metrics_calcs[f'y_pred_{staff_type}'], value['Mean'][:-19].to_numpy()))
                    train_metrics_calcs[f'lower_bound_{staff_type}'] = np.concatenate((train_metrics_calcs[f'lower_bound_{staff_type}'], value['Lower_95%'][:-19].to_numpy()))
                    train_metrics_calcs[f'upper_bound_{staff_type}'] = np.concatenate((train_metrics_calcs[f'upper_bound_{staff_type}'], value['Upper_95%'][:-19].to_numpy()))
                    train_metrics_calcs[f'y_true_{staff_type}'] = np.concatenate((train_metrics_calcs[f'y_true_{staff_type}'], value[f'y{group} NH No. 3 {staff_type}'][:-19].to_numpy()))
                    train_metrics_calcs[f'y_pred_{staff_type}'] = np.concatenate((train_metrics_calcs[f'y_pred_{staff_type}'], value['Mean'][:-19].to_numpy()))
                    train_metrics_calcs[f'lower_bound_{staff_type}'] = np.concatenate((train_metrics_calcs[f'lower_bound_{staff_type}'], value['Lower_95%'][:-19].to_numpy()))
                    train_metrics_calcs[f'upper_bound_{staff_type}'] = np.concatenate((train_metrics_calcs[f'upper_bound_{staff_type}'], value['Upper_95%'][:-19].to_numpy()))
                    train_metrics_calcs[f'y_true_{staff_type}'] = np.concatenate((train_metrics_calcs[f'y_true_{staff_type}'], value[f'y{group} NH No. 4 {staff_type}'][:-19].to_numpy()))
                    train_metrics_calcs[f'y_pred_{staff_type}'] = np.concatenate((train_metrics_calcs[f'y_pred_{staff_type}'], value['Mean'][:-19].to_numpy()))
                    train_metrics_calcs[f'lower_bound_{staff_type}'] = np.concatenate((train_metrics_calcs[f'lower_bound_{staff_type}'], value['Lower_95%'][:-19].to_numpy()))
                    train_metrics_calcs[f'upper_bound_{staff_type}'] = np.concatenate((train_metrics_calcs[f'upper_bound_{staff_type}'], value['Upper_95%'][:-19].to_numpy()))
                    train_metrics_calcs[f'y_true_{staff_type}'] = np.concatenate((train_metrics_calcs[f'y_true_{staff_type}'], value[f'y{group} NH No. 5 {staff_type}'][:-19].to_numpy()))
                    train_metrics_calcs[f'y_pred_{staff_type}'] = np.concatenate((train_metrics_calcs[f'y_pred_{staff_type}'], value['Mean'][:-19].to_numpy()))
                    train_metrics_calcs[f'lower_bound_{staff_type}'] = np.concatenate((train_metrics_calcs[f'lower_bound_{staff_type}'], value['Lower_95%'][:-19].to_numpy()))
                    train_metrics_calcs[f'upper_bound_{staff_type}'] = np.concatenate((train_metrics_calcs[f'upper_bound_{staff_type}'], value['Upper_95%'][:-19].to_numpy()))

                    # for test set (19 per group staff type)
                    test_metrics_calcs[f'y_true_{staff_type}'] = np.concatenate((test_metrics_calcs[f'y_true_{staff_type}'], value[f'y{group} NH No. 1 {staff_type}'].tail(19).to_numpy()))
                    test_metrics_calcs[f'y_pred_{staff_type}'] = np.concatenate((test_metrics_calcs[f'y_pred_{staff_type}'], value['Mean'].tail(19).to_numpy()))
                    test_metrics_calcs[f'lower_bound_{staff_type}'] = np.concatenate((test_metrics_calcs[f'lower_bound_{staff_type}'], value['Lower_95%'].tail(19).to_numpy()))
                    test_metrics_calcs[f'upper_bound_{staff_type}'] = np.concatenate((test_metrics_calcs[f'upper_bound_{staff_type}'], value['Upper_95%'].tail(19).to_numpy()))
                    test_metrics_calcs[f'y_true_{staff_type}'] = np.concatenate((test_metrics_calcs[f'y_true_{staff_type}'], value[f'y{group} NH No. 2 {staff_type}'].tail(19).to_numpy()))
                    test_metrics_calcs[f'y_pred_{staff_type}'] = np.concatenate((test_metrics_calcs[f'y_pred_{staff_type}'], value['Mean'].tail(19).to_numpy()))
                    test_metrics_calcs[f'lower_bound_{staff_type}'] = np.concatenate((test_metrics_calcs[f'lower_bound_{staff_type}'], value['Lower_95%'].tail(19).to_numpy()))
                    test_metrics_calcs[f'upper_bound_{staff_type}'] = np.concatenate((test_metrics_calcs[f'upper_bound_{staff_type}'], value['Upper_95%'].tail(19).to_numpy()))
                    test_metrics_calcs[f'y_true_{staff_type}'] = np.concatenate((test_metrics_calcs[f'y_true_{staff_type}'], value[f'y{group} NH No. 3 {staff_type}'].tail(19).to_numpy()))
                    test_metrics_calcs[f'y_pred_{staff_type}'] = np.concatenate((test_metrics_calcs[f'y_pred_{staff_type}'], value['Mean'].tail(19).to_numpy()))
                    test_metrics_calcs[f'lower_bound_{staff_type}'] = np.concatenate((test_metrics_calcs[f'lower_bound_{staff_type}'], value['Lower_95%'].tail(19).to_numpy()))
                    test_metrics_calcs[f'upper_bound_{staff_type}'] = np.concatenate((test_metrics_calcs[f'upper_bound_{staff_type}'], value['Upper_95%'].tail(19).to_numpy()))
                    test_metrics_calcs[f'y_true_{staff_type}'] = np.concatenate((test_metrics_calcs[f'y_true_{staff_type}'], value[f'y{group} NH No. 4 {staff_type}'].tail(19).to_numpy()))
                    test_metrics_calcs[f'y_pred_{staff_type}'] = np.concatenate((test_metrics_calcs[f'y_pred_{staff_type}'], value['Mean'].tail(19).to_numpy()))
                    test_metrics_calcs[f'lower_bound_{staff_type}'] = np.concatenate((test_metrics_calcs[f'lower_bound_{staff_type}'], value['Lower_95%'].tail(19).to_numpy()))
                    test_metrics_calcs[f'upper_bound_{staff_type}'] = np.concatenate((test_metrics_calcs[f'upper_bound_{staff_type}'], value['Upper_95%'].tail(19).to_numpy()))
                    test_metrics_calcs[f'y_true_{staff_type}'] = np.concatenate((test_metrics_calcs[f'y_true_{staff_type}'], value[f'y{group} NH No. 5 {staff_type}'].tail(19).to_numpy()))
                    test_metrics_calcs[f'y_pred_{staff_type}'] = np.concatenate((test_metrics_calcs[f'y_pred_{staff_type}'], value['Mean'].tail(19).to_numpy()))
                    test_metrics_calcs[f'lower_bound_{staff_type}'] = np.concatenate((test_metrics_calcs[f'lower_bound_{staff_type}'], value['Lower_95%'].tail(19).to_numpy()))
                    test_metrics_calcs[f'upper_bound_{staff_type}'] = np.concatenate((test_metrics_calcs[f'upper_bound_{staff_type}'], value['Upper_95%'].tail(19).to_numpy()))

    for staff_type in ['CNA', 'LPN', 'RN']:
        #populate the training metrics
        y_true = train_metrics_calcs[f'y_true_{staff_type}']
        y_pred = train_metrics_calcs[f'y_pred_{staff_type}']
        lower_bound = train_metrics_calcs[f'lower_bound_{staff_type}']
        upper_bound = train_metrics_calcs[f'upper_bound_{staff_type}']
        train_metrics_results = evaluate_predictions(y_true, y_pred, lower_bound, upper_bound)
        train_metrics[f'{staff_type}']['MAE'] = train_metrics_results['MAE']
        train_metrics[f'{staff_type}']['MAPE'] = train_metrics_results['MAPE']
        train_metrics[f'{staff_type}']['SMAPE'] = train_metrics_results['SMAPE']
        train_metrics[f'{staff_type}']['MIS'] = train_metrics_results['MIS']
        # Populate the test metrics
        y_true = test_metrics_calcs[f'y_true_{staff_type}']
        y_pred = test_metrics_calcs[f'y_pred_{staff_type}']
        lower_bound = test_metrics_calcs[f'lower_bound_{staff_type}']
        upper_bound = test_metrics_calcs[f'upper_bound_{staff_type}']
        test_metrics_results = evaluate_predictions(y_true, y_pred, lower_bound, upper_bound)
        test_metrics[f'{staff_type}']['MAE'] = test_metrics_results['MAE']
        test_metrics[f'{staff_type}']['MAPE'] = test_metrics_results['MAPE']
        test_metrics[f'{staff_type}']['SMAPE'] = test_metrics_results['SMAPE']
        test_metrics[f'{staff_type}']['MIS'] = test_metrics_results['MIS']
    # Print training metrics
    print("\nTraining Set Metrics:")
    print("--------------------")
    for staff_type in ['CNA', 'LPN', 'RN']:
        print(f"\n{staff_type} Metrics:")
        print("-" * 20)
        for metric_name in ['MAE', 'MAPE', 'SMAPE', 'MIS']:
            values = train_metrics[staff_type][metric_name]
            avg_value = np.mean(values)
            print(f"{metric_name}: {avg_value:.2f}")
    
    # Print test metrics
    print("\nTest Set Metrics:")
    print("----------------")
    for staff_type in ['CNA', 'LPN', 'RN']:
        print(f"\n{staff_type} Metrics:")
        print("-" * 20)
        for metric_name in ['MAE', 'MAPE', 'SMAPE', 'MIS']:
            values = test_metrics[staff_type][metric_name]
            avg_value = np.mean(values)
            print(f"{metric_name}: {avg_value:.2f}")

In [27]:
evaluate_all_data(groups)


Training Set Metrics:
--------------------

CNA Metrics:
--------------------
MAE: 30.47
MAPE: 25.97
SMAPE: 21.88
MIS: 91.59

LPN Metrics:
--------------------
MAE: 17.90
MAPE: 49.91
SMAPE: 37.51
MIS: 61.68

RN Metrics:
--------------------
MAE: 15.38
MAPE: 66.83
SMAPE: 53.56
MIS: 52.41

Test Set Metrics:
----------------

CNA Metrics:
--------------------
MAE: 30.92
MAPE: 24.66
SMAPE: 21.86
MIS: 90.61

LPN Metrics:
--------------------
MAE: 17.57
MAPE: 52.40
SMAPE: 38.14
MIS: 51.73

RN Metrics:
--------------------
MAE: 15.81
MAPE: 68.91
SMAPE: 52.10
MIS: 53.30


In [ ]:
# from ray import tune
# from ray.tune.search.hyperopt import HyperOptSearch
# from neuralforecast.auto import AutoTSMixer

# tsmixer_config = {
#        "input_size": input_size,                                                 # Size of input window
#        "max_steps": tune.choice([500, 1000, 2000]),                              # Number of training iterations
#        "val_check_steps": 100,                                                   # Compute validation every x steps
#        "early_stop_patience_steps": 5,                                           # Early stopping steps
#        "learning_rate": tune.loguniform(1e-4, 1e-2),                             # Initial Learning rate
#        "n_block": tune.choice([1, 2, 4, 6, 8]),                                  # Number of mixing layers
#        "dropout": tune.uniform(0.0, 0.99),                                       # Dropout
#        "ff_dim": tune.choice([32, 64, 128]),                                     # Dimension of the feature linear layer
#        "scaler_type": 'identity',       
#     }

# tsmixerx_config = tsmixer_config.copy()

# model = AutoTSMixer(h=horizon,
#                     n_series=300,
#                     loss=MAE(),
#                     config=tsmixer_config,
#                     num_samples=10,
#                     search_alg=HyperOptSearch(),
#                     backend='ray',
#                     valid_loss=MAE())

# nf = NeuralForecast(models=[model], freq='1D')
# Y_hat_df = nf.cross_validation(df=Y_df, val_size=val_size,
#                                test_size=test_size, n_windows=None)


In [ ]:
# nf.models[0].results.get_best_result().config

In [ ]:
# evaluate(Y_hat_df.drop(columns='cutoff'), metrics=[mae, mape], agg_fn="mean")